# Kimi Audio Few-Shot AD Detection

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"]     = "/root/autodl-tmp/LLM_Model"

import json
import tempfile
from pathlib import Path

import torch
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)

In [ ]:
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded.")

In [ ]:
MAX_DURATION = 60  # seconds
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert to 16kHz mono wav, truncated to MAX_DURATION seconds."""
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if wav_path.exists():
        return wav_path
    audio, sr = librosa.load(str(audio_path), sr=16000, mono=True, duration=MAX_DURATION)
    sf.write(str(wav_path), audio, sr)
    return wav_path

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)


def classify_audio(wav_path: Path, control_wavs: list, dementia_wavs: list) -> str:
    """Classify a single audio file with 3-shot examples (3 Control + 3 Dementia)."""
    messages = [
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[0])},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[1])},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(control_wavs[2])},
        {"role": "assistant", "message_type": "text",  "content": "Control"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[0])},
        {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[1])},
        {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
        {"role": "user",      "message_type": "audio", "content": str(dementia_wavs[2])},
        {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT},
        {"role": "user",      "message_type": "audio", "content": str(wav_path)},
    ]
    _, text = model.generate(messages, output_type="text")
    return text


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None

In [ ]:
OUTPUT_DIR = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result")


def get_examples(df, audio_dir, n=3):
    """Pick the first n Control and first n Dementia samples as few-shot examples."""
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    examples = {}
    for ad_val, label in label_map.items():
        rows = df[df["ad"] == ad_val].iloc[:n]
        wavs, ids = [], []
        for _, row in rows.iterrows():
            matches = list(audio_dir.glob(f"{label}/{row['session_id']}.*"))
            wavs.append(ensure_wav(matches[0]))
            ids.append(row["session_id"])
        examples[label] = {"session_ids": ids, "wavs": wavs}
    print(f"  Few-shot examples: Control={examples['Control']['session_ids']}, Dementia={examples['Dementia']['session_ids']}")
    return examples


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    # Select few-shot examples from this dataset
    examples = get_examples(df, audio_dir)
    example_ids = set(examples["Control"]["session_ids"] + examples["Dementia"]["session_ids"])
    control_wavs  = examples["Control"]["wavs"]
    dementia_wavs = examples["Dementia"]["wavs"]

    predictions, skipped = [], 0

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        # Skip few-shot example samples
        if row["session_id"] in example_ids:
            skipped += 1
            continue
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]), control_wavs, dementia_wavs)
            pred = parse_prediction(raw)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            raw, pred = "OOM", None
        except Exception as e:
            raw, pred = str(e), None
        finally:
            torch.cuda.empty_cache()
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred) * 100:.2f}%")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia') * 100:.2f}%")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1) * 100:.2f}%")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) * 100:.2f}%")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [ ]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")